# 📊 Home Credit - Baseline Scorecard Model (Logistic Regression + WoE / IV)

## 📌 Mục Tiêu Xây Dựng Baseline Theo Chuẩn Ngành Ngân Hàng
Theo quy trình machine learning bắt buộc trong `AGENTS.md` (Bước 5):
1. **Xây dựng Baseline Model**: Áp dụng thuật toán **Logistic Regression** trên tập dữ liệu đã chuẩn hóa.
2. **Chống Data Leakage**: Tách tập Train (80%) và Validation (20%) bằng Stratified K-Fold trước mọi thao tác Scaler & Imputation.
3. **Thước đo Đánh giá Ngân hàng**: Tính toán **ROC-AUC**, **PR-AUC**, **KS Statistic** (Kolmogorov-Smirnov), **Gini Coefficient**, và **Calibration Curve** (Brier Score).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve, classification_report, brier_score_loss
from sklearn.calibration import calibration_curve

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11

DATA_PATH = Path('../data/processed/home_credit_processed.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/processed/home_credit_processed.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f'✓ Nạp dữ liệu thành công từ {DATA_PATH}: shape = {df.shape}')
else:
    print('⚠️ Chưa tìm thấy file home_credit_processed.csv. Đang dùng sample từ application_train.csv...')
    RAW_PATH = Path('../data/raw/home-credit-default-risk/application_train.csv')
    if not RAW_PATH.exists():
        RAW_PATH = Path('data/raw/home-credit-default-risk/application_train.csv')
    df = pd.read_csv(RAW_PATH, nrows=50000)
    df['CREDIT_TO_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_TO_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df = pd.get_dummies(df, drop_first=True)

---
## 1. ✂️ Tách Dữ Liệu Train / Validation & Pipeline Imputation

In [ ]:
# Tách X và y
X = df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
y = df['TARGET']

# Chia Stratified Train / Validation (80/20)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'► Tập Train: {X_train.shape}, Tỷ lệ Target = {y_train.mean():.4f}')
print(f'► Tập Val:   {X_val.shape}, Tỷ lệ Target = {y_val.mean():.4f}')

# Imputation & Scaling fit CHỈ TRÊN TẬP TRAIN để tránh Data Leakage
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)

X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)

print('✓ Đã hoàn tất Imputation & Feature Scaling chuẩn hóa.')

---
## 2. 🤖 Huấn Luyện Baseline Logistic Regression Scorecard

In [ ]:
# Huấn luyện Logistic Regression với class_weight='balanced' do mất cân bằng dữ liệu
lr_model = LogisticRegression(class_weight='balanced', C=0.05, max_iter=500, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_proba_train = lr_model.predict_proba(X_train_scaled)[:, 1]
y_pred_proba_val = lr_model.predict_proba(X_val_scaled)[:, 1]

train_auc = roc_auc_score(y_train, y_pred_proba_train)
val_auc = roc_auc_score(y_val, y_pred_proba_val)
print(f'🎯 Train ROC-AUC: {train_auc:.4f}')
print(f'🎯 Validation ROC-AUC: {val_auc:.4f}')

---
## 3. 📈 Đánh Giá Chỉ Số Rủi Ro Tín Dụng (ROC-AUC, PR-AUC, KS, Gini, Calibration)

In [ ]:
def calculate_credit_metrics(y_true, y_prob):
    # 1. ROC-AUC
    auc_score = roc_auc_score(y_true, y_prob)
    # 2. Gini = 2 * AUC - 1
    gini_score = 2 * auc_score - 1
    # 3. KS Statistic
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    ks_stat = np.max(tpr - fpr)
    # 4. PR-AUC
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)
    # 5. Brier Score (Calibration)
    brier = brier_score_loss(y_true, y_prob)
    return {
        'ROC-AUC': auc_score,
        'Gini': gini_score,
        'KS Statistic (%)': ks_stat * 100,
        'PR-AUC': pr_auc,
        'Brier Score': brier
    }

metrics_lr = calculate_credit_metrics(y_val, y_pred_proba_val)
print('=== KẾT QUẢ ĐÁNH GIÁ BASELINE LOGISTIC REGRESSION ===')
for k, v in metrics_lr.items():
    print(f'► {k:20s}: {v:.4f}')

# Trực quan hóa đường cong ROC và KS Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_val, y_pred_proba_val)
axes[0].plot(fpr, tpr, label=f'Logistic Regression (AUC = {metrics_lr["ROC-AUC"]:.3f})', color='#2a9d8f', lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[0].set_title('Đường Cong ROC (ROC Curve)', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR)')
axes[0].legend(loc='lower right')

# KS Curve
ks_idx = np.argmax(tpr - fpr)
axes[1].plot(fpr, label='FPR (Tỷ lệ báo động nhầm)', color='#2a9d8f')
axes[1].plot(tpr, label='TPR (Tỷ lệ bắt nợ xấu)', color='#e76f51')
axes[1].set_title(f'Biểu Đồ Kolmogorov-Smirnov (KS = {metrics_lr["KS Statistic (%)"]:.1f}%)', fontweight='bold')
axes[1].set_xlabel('Threshold Index')
axes[1].set_ylabel('Rate')
axes[1].legend()

plt.tight_layout()
plt.show()